In [4]:
import math
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset, ConcatDataset

from torchvision import datasets, transforms

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from IPython.display import clear_output
import torch
from torch.utils.data import TensorDataset, DataLoader

from tqdm.notebook import tqdm

In [5]:
# def add_pattern(img):
    
#     if count % 3 == 0:
#         return add_diag_stripes(img)
#     elif count % 3 == 1:
#         return add_diamonds(img)
#     elif count % 3 == 2:
#         return add_wave(img)

def add_diag_stripes(img):
    period, center, thickness = random.randint(4, 10), random.randint(0, 4), random.randint(0, 2)
    
    i = torch.arange(28).unsqueeze(1)
    j = torch.arange(28).unsqueeze(0)
    
    x = torch.ones(28, 28)
    mask = ((i - j - center) % period).abs() <= thickness
    x[mask] = random.randint(40, 80)/100
    return img * x

def add_diamonds(img):
    ci, cj, thickness = random.randint(10, 17), random.randint(10, 17), random.randint(0, 3)
    
    i = torch.arange(28).unsqueeze(1)
    j = torch.arange(28).unsqueeze(0)
    
    x = torch.ones(28, 28)
    x[((torch.abs(i - ci) + torch.abs(j - cj)) % 5 <= thickness)] = random.randint(40, 80)/100
    return img * x

def add_wave(img):
    amplitude, frequency, height, thickness, phase = random.randint(4, 10), random.randint(1, 4), random.randint(3, 20), random.randint(1, 3), random.uniform(0, 2 * math.pi)
    
    i = torch.arange(28)
    j = torch.arange(28)
    curve = height + amplitude * torch.cos(2 * math.pi * frequency * j / 28 + phase)
    
    rows = curve.round().long()
    x = torch.ones(28, 28)
    for d in range(-thickness, thickness + 1):
        r = (rows + d).clamp(0, 27)
        x[r, j] = 0.5
    return img * x

In [6]:
pre_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Normalize((0.5,), (0.5,))
])

train_base = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=pre_transform,
    download=True
)

test_base = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=pre_transform,
    download=True
)

train_dataset = PatternDataset(train_base, True)
test_dataset  = PatternDataset(test_base, False)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

In [7]:
class PatternDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, train=True):
        self.base = base_dataset
        self.train = train

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, y = self.base[idx]

        p = torch.randint(0, 3, ()).item()
        if p == 0:
            img = add_wave(img[0]).unsqueeze(0)
            y = y
        elif p == 1:
            img = add_diamonds(img[0]).unsqueeze(0)
            y = y + 10
        else:
            img = add_diag_stripes(img[0]).unsqueeze(0)
            y = y + 20

        if self.train == True:
            return train_transform(img), y
        else:
            return test_transform(img), y

In [8]:
pre_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Normalize((0.5,), (0.5,))
])

train_base = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=pre_transform,
    download=True
)

test_base = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=pre_transform,
    download=True
)

train_dataset = PatternDataset(train_base, True)
test_dataset  = PatternDataset(test_base, False)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

In [9]:
xb, yb = next(iter(train_loader))
print(xb.shape, xb.min(), xb.max(), xb.mean(), xb.std())
print(yb.unique())


torch.Size([128, 1, 28, 28]) tensor(-1.) tensor(1.) tensor(-0.5076) tensor(0.6446)
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])


In [10]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.GroupNorm(16, 64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GroupNorm(16, 128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GroupNorm(16, 256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.fc = nn.Linear(256, 30)

    def forward(self, x):
        x = self.features(x)
        return self.fc(x)


cls = Classifier()
print(sum(p.numel() for p in cls.parameters()))

378270


In [11]:
cls(xb).shape

torch.Size([128, 30])

In [12]:
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import argparse
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cls = Classifier().to(device)

pre_epoch = 0
BATCH_SIZE = 128
LR = 0.001

epochs = 50
pre_epoch = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    cls.parameters(),
    lr=LR,
    weight_decay=0.001
)

In [13]:
best_acc = 0.0
best_model = None

for epoch in range(epochs):
    cls.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cls(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

        pbar.set_postfix({
            "Loss": f"{running_loss / len(pbar):.3f}",
            "Acc": f"{100. * correct / total:.2f}%"
        })

    if epoch % 10 == 0:
        torch.save(best_model, f"best_fashion_mnist_cnn_{epoch}.pth")

    # -------------------------
    # Evaluation
    # -------------------------
    cls.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = cls(inputs)
            _, preds = outputs.max(1)
            test_total += labels.size(0)
            test_correct += preds.eq(labels).sum().item()

    test_acc = 100. * test_correct / test_total
    print(f"--- Epoch {epoch+1} | Test Acc: {test_acc:.2f}% ---")

    if test_acc > best_acc:
        best_acc = test_acc
        best_model = cls.state_dict()

Epoch 1/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 1 | Test Acc: 55.52% ---


Epoch 2/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 2 | Test Acc: 65.75% ---


Epoch 3/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 3 | Test Acc: 72.10% ---


... (16 outputs left out)

Epoch 12/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 12 | Test Acc: 83.09% ---


Epoch 13/50:   0%|          | 0/469 [00:00<?, ?it/s]


KeyboardInterrupt



In [14]:
torch.save(best_model, f"fashion_pattern_mnist_cnn.pth")